# 02 — Core DB setup (UDA-Hub)

This notebook initializes and seeds `data/core/udahub.db` — the database owned by **UDA-Hub itself**: `tickets`, `ticket_metadata`, `ticket_messages`, `knowledge`, `long_term_memory`, and `agent_run_log`.

The `knowledge` table is UDA-Hub's copy of the knowledge base CultPass handed off (conceptually `cultpass_articles.jsonl` in the brief). This project had no starter repo to copy an original 4-article file from (see the top-level README), so all **18** articles in `data/core/seed_knowledge.py` were authored fresh for this submission, spanning 10 categories — comfortably past the "at least 14 total / 10 additional" requirement.

Run all cells top to bottom. It's safe to re-run: seeding is skipped if the table already has data.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
print("Core DB path:", config.CORE_DB_PATH)

In [ ]:
from data.core.seed_knowledge import seed

seed()

## Verify the data

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(config.CORE_DB_PATH)

df = pd.read_sql("SELECT id, title, category, tags FROM knowledge ORDER BY category, id", conn)
print(f"{len(df)} knowledge articles across {df['category'].nunique()} categories:")
display(df)

print("\nArticles per category:")
display(df.groupby("category").size().rename("count"))

conn.close()

## Build the RAG index and test retrieval

This calls the actual `kb_search_tool` the agents use (see `agentic/tools/kb_search_tool.py`) — it will build and cache a FAISS index under `data/models/knowledge_index/` on first run (requires `OPENAI_API_KEY` in `.env`).

In [ ]:
from agentic.tools.kb_search_tool import kb_search_tool

result = kb_search_tool.invoke({"query": "Can I get my money back for a booking I made last week?", "k": 3})
for r in result["results"]:
    print(f"[{r['score']:.3f}] {r['title']}  ({r['category']})")

In [ ]:
# A query with no good match should surface a low top_score — this is what
# the Resolver agent's escalation gate keys off of (CONFIDENCE_THRESHOLD in config.py).
result = kb_search_tool.invoke({"query": "Can you help me plan a birthday party with catering?", "k": 3})
print("top_score:", result["top_score"])
for r in result["results"]:
    print(f"[{r['score']:.3f}] {r['title']}")